# Metric-corrected edits — is the probe's row space `J` *whitened*, and does un-whitening recover it?

**Thread:** `editability/` · **Metric/editor definitions:** `../METRICS_AND_EDITORS.md` · **Conventions:**
`../../../../CLAUDE.md`. **Model:** GRU `runs/controls/H256` (H=256) — the checkpoint the thread's findings
were established on. **Dataset:** `datasets/4_fixed_refl_inview`; `test` split for the state bank and probe,
`edits` split (N=256) for every edit. **No retraining.**

## The motivation (Sevan's derivation)

A least-squares linear probe is

$$W \;=\; \Sigma_{ph}\,\Sigma_{hh}^{-1}$$

Suppose the local structure is `h ≈ h₀ + Jp`, where `J = ∂h/∂p` is **the direction the hidden state actually
moves when position changes** — the thing an editor needs. Then `Σ_ph = Σ_pp Jᵀ`, so

$$W^{\top} \;=\; \Sigma_{hh}^{-1} J\,\Sigma_{pp} \qquad\Longrightarrow\qquad \operatorname{row}(W) \;=\; \operatorname{span}\!\left(\Sigma_{hh}^{-1} J\right)$$

**The probe's row space is `J` whitened by the inverse state covariance, not `J` itself.** If `Σ_hh` is strongly
anisotropic the two can be nearly orthogonal *even with a perfectly accurate probe* — which would explain,
without any appeal to representation quality, why every probe-derived edit direction in this thread has failed.

**Hypothesis.** This is the mechanism. Un-whitening — multiplying by `Σ_hh` — should recover the true direction.

**A prior that already points this way.** `../iterative_probing/` found that the linear position code sits in
**below-average-variance** directions of the state (63.4% of state energy retained after removing the 116
position dimensions, versus 56.9% for a matched random removal). That is exactly what `Σ_hh^{-1}` does: it
amplifies low-variance directions, so a least-squares probe preferentially reads them. The derivation and that
measurement agree.

## The four tests, and what each can conclude

| test | what it does | what it can conclude |
|---|---|---|
| **Gate** | eigenspectrum and condition number of `Σ_hh` | **if the condition number is small (≲20) the hypothesis is likely wrong**, and that is worth knowing before anything else |
| **1 — direction** | cosine of `W⁺δ` and of `Σ_hh^α W⁺δ` against `Δh_true`, for α ∈ {0, ¼, ½, ¾, 1} | does un-whitening point the edit in the right direction? No editing involved. |
| **2 — closed-form editor** | `Δ = Σ_hh Wᵀ(W Σ_hh Wᵀ + εI)⁻¹ δ`, then roll out and score | the Mahalanobis-metric version of the Euclidean pseudoinverse edit — **apples-to-apples** with the existing failing editor |
| **3 — scale sweep** | multiply the best direction by {0.5, 1, 2, 3, 4, 6, 8} | separates **"wrong direction"** from **"right direction, wrong magnitude"** — a distinction no earlier editor could make |
| **4 — local metric** | repeat 1 and 2 with `Σ_hh` estimated from nearest neighbours of `h₀` across trajectories | a global metric assumes one covariance everywhere; path-independence and superposition are *local* properties |

**Reading guide (agreed before running).**
- **Big cosine jump + working edits** → the mechanism is confirmed.
- **Cosine jump, edits still fail** → magnitude or manifold reachability; the scale sweep adjudicates.
- **No jump despite a large condition number** → the whitening hypothesis is **falsified**, and the remaining
  explanations narrow to the target being unreachable from readout-derived subspaces at all.

## One thing to get right about the α family

`Σ^α W⁺δ` (the literal shrinkage Sevan specified) **does not satisfy the readout constraint** for α ∉ {0}:
`W(Σ^α W⁺δ) ≠ δ`. It is a fine object for a *direction* test, which is all Test 1 needs. For **editing** we use
the constraint-satisfying family

$$\Delta_\alpha \;=\; \Sigma^{\alpha} W^{\top}\bigl(W \Sigma^{\alpha} W^{\top} + \varepsilon I\bigr)^{-1}\delta$$

which hits the readout target **exactly at every α**, and interpolates from the Euclidean pseudoinverse (α=0,
reproducing the thread's existing failing editor) to the full Mahalanobis solution (α=1, Test 2). Both families
are reported in Test 1 so the specification is answered directly and the editing tests stay well-posed.

## Definitions — read this before any number

| symbol | meaning |
|---|---|
| `Σ_hh` | empirical covariance of **mean-centred** hidden states over a bank of on-manifold states from real rollouts, `(256, 256)` |
| `W`, `b` | the position probe, `p = Wh + b`, `W` of shape `(4, 256)`; target is `(x₀, y₀, x₁, y₁)` in sim units |
| `δ` | `target − (W h₀ + b)` — the readout error the editor is asked to close |
| `W⁺δ` | the **existing failing editor**: Euclidean minimum-norm solution of `WΔ = δ` |
| `Δ_α` | `Σ^α Wᵀ(W Σ^α Wᵀ + εI)⁻¹ δ` — the metric-corrected editor; α=0 is Euclidean, α=1 is Mahalanobis |
| `Δh_true` | a **successful** edit as a displacement: `h_oracle − h₀`, from **both** oracles (counterfactual state overwrite and freeze-time teacher forcing) |
| `J` | `∂h/∂p`, the direction `h` actually moves when position changes — the thing the derivation says `Σ_hh Wᵀ` recovers |

| metric | formula | units | better |
|---|---|---|---|
| **cosine to `Δh_true`** | `⟨u, Δh_true⟩ / (‖u‖‖Δh_true‖)`, **per sample then averaged** | — | ↑ |
| **angle** | `arccos` of the above | degrees | ↓ |
| **subspace mass fraction** | `‖P_S Δh_true‖ / ‖Δh_true‖`, `S` = the editor's 4-dim reachable subspace | fraction | ↑ vs chance |
| **chance for the mass fraction** | `√(d/H)` = `√(4/256)` = **0.125** — what a *random* vector already has | fraction | — |
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)` on differing rays, per sample then averaged | −1…+1 | ↑ |
| **Target / Ghost / Collateral / GT-traj RMSE** | canonical §4 zone errors at rollout step 0 (Edit Index step 0 decodes frame `ef`) | obs intensity | ↓ |
| **fidelity ratio** | `GT-traj RMSE(editor) / GT-traj RMSE(unsteered)` | ratio | ↓ — **>1 means the edit made things worse than doing nothing** |
| **‖Δ‖ / one dynamics step** | `‖Δ‖ ÷ mean‖h_t − h_{t−1}‖` over real rollouts | ratio | — |

### Baselines that every number must be read against

1. **The existing failing editor is α=0.** Its cosine to `Δh_true` is ≈ **0.08** and its Edit Index ≈ **−0.66**
   against an unsteered floor of **−0.67**. Any claimed improvement is measured from there, not from zero.
2. **Chance for a 4-dim subspace is 0.125**, not 0. A mass fraction of 0.15 is barely above chance, not "15%
   captured, mostly missing".
3. **A cosine is not a correlation.** cos 0.5 is **60°**; two equal-length vectors 60° apart differ by a full
   `2·sin(30°) = 1.0` of their length. Angles are reported beside every cosine.
4. **The Edit Index must be read against this model's own unsteered row**, and **no success claim survives a
   fidelity ratio > 1** — an index that moved because the output degraded into garbage reads ≈0 by design.

### Implementation details a reader would ask about

- `ef = 20`, `K_ROLL = 15`, `N_FT = 8` freeze-time frames, `N_EVAL = 256` held-out edits, bank = 2,000 `test`
  sequences × 39 aligned timesteps = 78,000 states, **float64** throughout the linear algebra.
- `Σ^α` via eigendecomposition `Σ = VΛVᵀ → VΛ^αVᵀ`, eigenvalues floored at `λ_max·1e-12`.
- `ε` in the 4×4 inverse is `1e-6 · trace/4` — reported, and the inverse is well-conditioned without it.
- **Local metric (Test 4):** `k = 1024` nearest neighbours of `h₀` **in hidden-state space across all
  trajectories in the bank** (not a temporal window around `h₀`), covariance centred on the neighbours' own
  mean. `k > 256` so the local covariance is generically full rank.
- Every observation-space error is scored against the **clean** render, never the noisy observation.

In [ ]:
# [1] Setup: model, data, the aligned state bank, the probe W, both oracles, and the §4 ray zones.
import os, sys, time
sys.path.insert(0, "../../../..")
sys.path.insert(0, "../../../../scripts")
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

from pim.world_models import load_checkpoint, load_dataset
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene
from pim.figures.theme import style_ax
from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ, K_ROLL, N_FT, N_EVAL, NBANK = 2, 15, 8, 256, 2000
OUT = "figures"; os.makedirs(OUT, exist_ok=True)
ROOT = "../../../.."

MODEL, INFO = load_checkpoint(f"{ROOT}/runs/controls/H256/best_model.pt", device=DEVICE)
Hd = MODEL.hidden_size
bundle = load_dataset(f"{ROOT}/datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
ef = edits.edit_frame; sim = test.config["dataset"]["sim"]; R = edits.obs_res; DT = float(sim["dt"])
with h5py.File(edits.h5_path, "r") as f:
    VEL_ALL = f["velocities"][:, :, :N_OBJ, :].astype(np.float32)

@torch.no_grad()
def aligned_bank(obs_np, chunk=500):
    """States aligned so decode(bank[:, t]) ↔ sim frame t+1 — the states this thread edits."""
    outs = []
    for s in range(0, len(obs_np), chunk):
        o = torch.from_numpy(obs_np[s:s + chunk]).float().to(DEVICE); st = None; seq = []
        for t in range(o.shape[1] - 1):
            _, st = MODEL.step(o[:, t], st); seq.append(MODEL.flat_state(st))
        outs.append(torch.stack(seq, 1).cpu().numpy())
    return np.concatenate(outs, 0)

@torch.no_grad()
def warm(obs_np, upto):
    o = torch.from_numpy(obs_np).float().to(DEVICE); st = None
    for t in range(upto):
        _, st = MODEL.step(o[:, t], st)
    return MODEL.flat_state(st)

@torch.no_grad()
def roll(h_flat, steps=K_ROLL):
    """Free-run; out[:, 0] = decode(h) = sim frame ef."""
    st = MODEL.state_from_flat(torch.as_tensor(h_flat, dtype=torch.float32, device=DEVICE))
    out = [MODEL.decode(st)]
    for _ in range(steps - 1):
        p, st = MODEL.predict_step(st); out.append(p)
    return torch.stack(out, 1).cpu().numpy()

t0 = time.perf_counter()
BANK3 = aligned_bank(test.obs[:NBANK].astype(np.float32))          # (N, T-1, H)
T_use = BANK3.shape[1]
H_BANK = BANK3.reshape(-1, Hd).astype(np.float64)
POS_BANK = test.positions[:NBANK, 1:1 + T_use, :N_OBJ, :].reshape(-1, N_OBJ * 2).astype(np.float64)
# reference scale for the sweep: how big is ONE ordinary dynamics step?
STEP_NORM = float(np.linalg.norm(np.diff(BANK3, axis=1), axis=-1).mean())
print(f"bank: {H_BANK.shape[0]:,} states x {Hd} dims in {time.perf_counter()-t0:.0f}s | "
      f"mean ‖h_t − h_(t−1)‖ = {STEP_NORM:.3f}")

# ── the probe W (fit on 80% of SEQUENCES, scored on the held-out 20%) ─────────
ntr = int(0.8 * NBANK) * T_use
Aug = np.concatenate([H_BANK[:ntr], np.ones((ntr, 1))], 1)
sol, *_ = np.linalg.lstsq(Aug, POS_BANK[:ntr], rcond=None)
W = sol[:-1].T.copy(); b_pr = sol[-1].copy()
pred_ho = H_BANK[ntr:] @ sol[:-1] + sol[-1]
R2_W = float(1 - ((pred_ho - POS_BANK[ntr:]) ** 2).sum() /
             ((POS_BANK[ntr:] - POS_BANK[:ntr].mean(0)) ** 2).sum())
W_PINV = np.linalg.pinv(W)
print(f"probe W: shape {W.shape}, held-out position R² = {R2_W:.3f}")

# ── the edit cases, both oracles, and the canonical zones ─────────────────────
REFL = np.array([sim["refl_min"], sim["refl_max"]], np.float32)
RAD  = np.array([sim["radius"]] * N_OBJ, np.float32)
COL  = np.tile(np.array([[1, 1, 1]], np.float32), (N_OBJ, 1))
def _cfg(nf, noise):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                     n_objects=N_OBJ, radius=sim["radius"], n_frames=nf, dt=sim["dt"], obs_res=sim["obs_res"],
                     refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                     obs_noise_std=noise, boundary="open", always_in_frustum=False)
def render_traj(pos_seq, noise=0.0):
    _, _, inten = render_scene(Scene(positions=pos_seq, velocities=np.zeros_like(pos_seq), radii=RAD,
                                     colors=COL, reflectivities=REFL, config=_cfg(len(pos_seq), noise)))
    return inten.astype(np.float32)

IDX = np.arange(N_EVAL)
oe  = edits.edit_object[IDX].astype(int)
pos = edits.positions[IDX][:, :, :N_OBJ, :].astype(np.float32)
TGT, PRE = pos[:, ef].copy(), pos[:, ef - 1]
OBS_NOISE = float(sim["obs_noise_std"])

cf_obs = np.zeros((N_EVAL, ef, R), np.float32)
ft_obs = np.zeros((N_EVAL, N_FT, R), np.float32)
t_idx = np.arange(ef)
for i in range(N_EVAL):
    o, other = oe[i], 1 - oe[i]
    v = VEL_ALL[IDX[i], ef, o]
    cf = np.zeros((ef, N_OBJ, 2), np.float32)
    cf[:, o]     = TGT[i, o][None, :] - v[None, :] * (ef - t_idx)[:, None] * DT
    cf[:, other] = pos[i, :ef, other]
    cf_obs[i] = render_traj(cf)
    fr = np.zeros((N_FT, N_OBJ, 2), np.float32)
    for j in range(N_FT):
        fr[j, o] = PRE[i, o] + ((j + 1) / N_FT) * (TGT[i, o] - PRE[i, o]); fr[j, other] = TGT[i, other]
    ft_obs[i] = render_traj(fr, OBS_NOISE)

H0 = warm(edits.obs[:N_EVAL].astype(np.float32), ef)
@torch.no_grad()
def continue_from(h_flat, frames):
    st = MODEL.state_from_flat(h_flat); o = torch.from_numpy(frames).float().to(DEVICE)
    for t in range(frames.shape[1]):
        _, st = MODEL.step(o[:, t], st)
    return MODEL.flat_state(st)
H_CF = warm(cf_obs, ef)
H_FT = continue_from(H0, ft_obs)

h0 = H0.cpu().numpy().astype(np.float64)
DH = {"counterfactual state overwrite": (H_CF - H0).cpu().numpy().astype(np.float64),
      "freeze-time teacher forcing":    (H_FT - H0).cpu().numpy().astype(np.float64)}
tgt4 = TGT.reshape(N_EVAL, N_OBJ * 2).astype(np.float64)
DELTA = tgt4 - (h0 @ W.T + b_pr)                                   # the readout error the editor must close

gt_roll = edits.clean_obs[:N_EVAL, ef:ef + K_ROLL, :].astype(np.float32)
ZONES = build_edit_zones(pre_pos=PRE, tgt_pos=TGT, pre_vel=VEL_ALL[IDX, ef - 1, :N_OBJ, :],
                         edit_object=oe, sim=sim, n_obj=N_OBJ,
                         traj_pos=edits.positions[:N_EVAL, ef:ef + K_ROLL, :N_OBJ, :].astype(np.float32),
                         gt_edited_traj=gt_roll)
n0 = np.linalg.norm(h0, axis=-1).mean()
for k, v in DH.items():
    print(f"‖Δh_true‖ ({k}): {np.linalg.norm(v,axis=-1).mean():.2f} = {np.linalg.norm(v,axis=-1).mean()/n0:.2f}·‖h0‖ "
          f"= {np.linalg.norm(v,axis=-1).mean()/STEP_NORM:.1f}× one dynamics step")

In [ ]:
# [2] THE GATE — eigenspectrum and condition number of Σ_hh.
# If Σ_hh is close to isotropic (condition number under ~20) then row(W) ≈ span(J) already,
# un-whitening can change little, and the hypothesis is likely wrong. Check before going further.
MU_H = H_BANK.mean(0)
Hc = H_BANK - MU_H
SIGMA = (Hc.T @ Hc) / (Hc.shape[0] - 1)
lam, V = np.linalg.eigh(SIGMA)                       # ascending
lam = lam[::-1]; V = V[:, ::-1]
lam_pos = np.clip(lam, lam[0] * 1e-12, None)
cond = float(lam[0] / lam[-1]) if lam[-1] > 0 else np.inf
cond_eff = float(lam[0] / lam_pos[-1])
evr = np.cumsum(lam) / lam.sum()
n90, n95, n99 = (int(np.searchsorted(evr, p) + 1) for p in (0.90, 0.95, 0.99))

def sigma_pow(a, eigval=lam_pos, eigvec=V):
    """Σ^a via eigendecomposition, eigenvalues floored at λ_max·1e-12."""
    return (eigvec * (eigval ** a)) @ eigvec.T

display(Markdown(
    f"**Gate — is `Σ_hh` anisotropic enough for whitening to matter?**\n\n"
    f"| quantity | value |\n|---|---|\n"
    f"| condition number `λ_max/λ_min` | **{cond:.3g}** |\n"
    f"| λ_max | {lam[0]:.4g} |\n| λ_min | {lam[-1]:.4g} |\n"
    f"| PCA dims to 90 / 95 / 99% variance | {n90} / {n95} / {n99} of {Hd} |\n"
    f"| top eigenvalue's share of total variance | {100*lam[0]/lam.sum():.1f}% |\n\n"
    + (f"**GATE PASSED** — condition number {cond:.3g} ≫ 20. `Σ_hh` is strongly anisotropic, so "
       f"`row(W) = span(Σ_hh⁻¹J)` and `span(J)` can differ a lot. The hypothesis is worth testing."
       if cond > 20 else
       f"**GATE FAILED** — condition number {cond:.3g} < 20. `Σ_hh` is close to isotropic, so `row(W)` ≈ "
       f"`span(J)` already and un-whitening cannot change much. The whitening hypothesis is likely wrong.")))

plt.style.use("default")
fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.3))
ax[0].semilogy(np.arange(1, Hd + 1), lam_pos, "o-", color="#0072B2", lw=1.6, ms=3)
ax[0].set_xlabel("eigenvalue index"); ax[0].set_ylabel("eigenvalue of Σ_hh (log scale)")
ax[0].set_title(f"(a) spectrum — condition number {cond:.3g}", fontsize=10)
ax[0].grid(alpha=0.3); style_ax(ax[0])
ax[1].plot(np.arange(1, Hd + 1), 100 * evr, "-", color="#009E73", lw=2.0)
for n, p, c in [(n90, 90, "#0072B2"), (n95, 95, "#CC79A7"), (n99, 99, "#D55E00")]:
    ax[1].axvline(n, color=c, ls="--", lw=1.2)
    ax[1].annotate(f"{p}%: {n}", xy=(n, 40), fontsize=7.5, color=c, rotation=90, ha="right", va="bottom")
ax[1].set_xlabel("number of components"); ax[1].set_ylabel("cumulative variance explained (%)")
ax[1].set_title("(b) how concentrated the state distribution is", fontsize=10)
ax[1].grid(alpha=0.3); style_ax(ax[1])
fig.suptitle("Fig 1 — the state covariance Σ_hh, which the probe implicitly inverts", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_sigma_spectrum.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [3] TEST 1 — direction check. No editing: just how well each candidate direction points at Δh_true.
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]
EPS_SCALE = 1e-6

def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), 1e-300)

def cos_to(u, v):
    return (unit(u) * unit(v)).sum(-1)

def delta_alpha(a, delta=DELTA, sig_pow=None):
    """Δ_α = Σ^α Wᵀ (W Σ^α Wᵀ + εI)⁻¹ δ — hits the readout target EXACTLY at every α."""
    S = sigma_pow(a) if sig_pow is None else sig_pow
    M = W @ S @ W.T
    M = M + EPS_SCALE * np.trace(M) / M.shape[0] * np.eye(M.shape[0])
    # rows: Δ = (Σ Wᵀ M⁻¹ δ)ᵀ = δᵀ M⁻¹ (W Σ), using M symmetric
    return (np.linalg.solve(M, delta.T).T) @ (W @ S)

def sigma_pow_pinv(a):
    """The literal Σ^α W⁺δ family Sevan specified — a DIRECTION test only; it does not satisfy WΔ = δ."""
    return (DELTA @ W_PINV.T) @ sigma_pow(a).T

DIRS = {}
for a in ALPHAS:
    DIRS[("constraint-satisfying  Δ_α", a)] = delta_alpha(a)
    DIRS[("literal  Σ^α W⁺δ", a)] = sigma_pow_pinv(a)

# subspace mass fraction: the editor's reachable 4-dim subspace is span(Σ^α Wᵀ)
def mass_frac(a, dh):
    B = np.linalg.qr(sigma_pow(a) @ W.T)[0]                  # (256, 4) orthonormal
    return float((np.linalg.norm(dh @ B, axis=-1) / np.linalg.norm(dh, axis=-1)).mean())

CHANCE4 = float(np.sqrt(W.shape[0] / Hd))
rows = ["| Δh_true from | family | α | cosine to Δh_true | angle | subspace mass fraction | ÷ chance (0.125) |",
        "|---|---|---|---|---|---|---|"]
T1 = {}
for onm, dh in DH.items():
    for fam in ["constraint-satisfying  Δ_α", "literal  Σ^α W⁺δ"]:
        for a in ALPHAS:
            c = cos_to(DIRS[(fam, a)], dh)
            mf = mass_frac(a, dh)
            T1[(onm, fam, a)] = dict(cos=float(c.mean()), sd=float(c.std()), mass=mf)
            rows.append(f"| {onm} | {fam} | {a:.2f} | **{c.mean():+.3f}** ± {c.std():.3f} | "
                        f"{np.degrees(np.arccos(np.clip(c.mean(),-1,1))):.0f}° | {mf:.3f} | {mf/CHANCE4:.2f}× |")
display(Markdown(
    "**Table 1 — Test 1: does un-whitening point the edit in the right direction?** Cosines are per sample then "
    "averaged (± sd across samples). **α = 0 is the thread's existing failing editor**, so it is the baseline "
    "every other row is measured against. The subspace mass fraction is `‖P_S·Δh_true‖/‖Δh_true‖` for the "
    "editor's reachable 4-dim subspace `span(Σ^α Wᵀ)`, directly comparable to the thread's existing "
    f"orthogonality numbers; chance for any 4-dim subspace is `√(4/256)` = **{CHANCE4:.3f}**.\n\n" + "\n".join(rows)))

fig, ax = plt.subplots(1, 2, figsize=(14.5, 4.5))
CO = {"counterfactual state overwrite": "#0072B2", "freeze-time teacher forcing": "#D55E00"}
for onm in DH:
    for fam, ls, mk in [("constraint-satisfying  Δ_α", "-", "o"), ("literal  Σ^α W⁺δ", "--", "s")]:
        ax[0].plot(ALPHAS, [T1[(onm, fam, a)]["cos"] for a in ALPHAS], ls, marker=mk, color=CO[onm],
                   lw=2.0, ms=5, label=f"{onm} · {fam}")
ax[0].axhline(0, color="0.4", ls=":", lw=1.0)
ax[0].set_xlabel("α  (0 = Euclidean / the existing editor,  1 = full un-whitening)")
ax[0].set_ylabel("cosine to Δh_true")
ax[0].set_title("(a) direction agreement with a SUCCESSFUL edit", fontsize=10)
ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3); style_ax(ax[0])

for onm in DH:
    ax[1].plot(ALPHAS, [T1[(onm, "constraint-satisfying  Δ_α", a)]["mass"] for a in ALPHAS], "-o",
               color=CO[onm], lw=2.0, ms=5, label=onm)
ax[1].axhline(CHANCE4, color="0.35", ls="--", lw=1.4)
ax[1].annotate(f"chance for any 4-dim subspace: {CHANCE4:.3f}", xy=(1.0, CHANCE4), fontsize=7.5,
               color="0.35", ha="right", va="bottom")
ax[1].set_xlabel("α"); ax[1].set_ylabel("‖P_S·Δh_true‖ / ‖Δh_true‖")
ax[1].set_title("(b) how much of a successful edit the reachable\nsubspace span(Σ^α Wᵀ) contains", fontsize=10)
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3); style_ax(ax[1])
fig.suptitle("Fig 2 — Test 1: does un-whitening recover the true edit direction?", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_direction_check.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [4] TEST 2 — the closed-form metric-corrected editor, scored on the canonical §4 metrics.
def score(dh_edit, label, cards):
    rl = roll(h0 + dh_edit)
    c = edit_scorecard(rl, ZONES, gt_roll)
    c["fidelity_ratio"] = 1.0 if label == "unsteered (no edit)" else fidelity_ratio(c, cards["unsteered (no edit)"])
    c["mag_h0"] = float((np.linalg.norm(dh_edit, axis=-1) / n0).mean())
    c["mag_step"] = float(np.linalg.norm(dh_edit, axis=-1).mean() / STEP_NORM)
    c["roll"] = rl
    cards[label] = c
    return c

CARDS = {}
score(np.zeros_like(h0), "unsteered (no edit)", CARDS)
for a in ALPHAS:
    lab = ("readout injection — Euclidean, α=0 (the existing failing editor)" if a == 0
           else f"metric-corrected Δ_α, α={a:.2f}" + (" (full Mahalanobis)" if a == 1 else ""))
    score(delta_alpha(a), lab, CARDS)
CARDS["counterfactual state overwrite (ORACLE)"] = edit_scorecard(roll(h0 + DH["counterfactual state overwrite"]), ZONES, gt_roll)
CARDS["counterfactual state overwrite (ORACLE)"]["fidelity_ratio"] = fidelity_ratio(
    CARDS["counterfactual state overwrite (ORACLE)"], CARDS["unsteered (no edit)"])
CARDS["counterfactual state overwrite (ORACLE)"]["mag_h0"] = float((np.linalg.norm(DH["counterfactual state overwrite"], axis=-1) / n0).mean())
CARDS["counterfactual state overwrite (ORACLE)"]["mag_step"] = float(np.linalg.norm(DH["counterfactual state overwrite"], axis=-1).mean() / STEP_NORM)
CARDS["counterfactual state overwrite (ORACLE)"]["roll"] = roll(h0 + DH["counterfactual state overwrite"])

ORDER2 = list(CARDS)
rows = ["| editor | ‖Δ‖/‖h0‖ | ‖Δ‖ ÷ one dynamics step | Edit Index (step 0) ↑ | at step 14 | Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | GT-traj RMSE ↓ | fidelity ↓ |",
        "|---|---|---|---|---|---|---|---|---|---|"]
for lab in ORDER2:
    c = CARDS[lab]
    rows.append(f"| {lab} | {c['mag_h0']:.3f} | {c['mag_step']:.2f}× | **{c['edit_index']:+.2f}** | "
                f"{c['edit_index_by_step'][-1]:+.2f} | {c['target_rmse']:.3f} | {c['ghost_rmse']:.3f} | "
                f"{c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} | {c['fidelity_ratio']:.2f} |")
display(Markdown("**Table 2 — Test 2: does the metric-corrected edit actually work?** `Δ_α` hits the readout "
                 "target exactly at every α, so this is **apples-to-apples** with the existing Euclidean "
                 "pseudoinverse editor (α=0) — the *only* difference is the metric. Read the Edit Index against "
                 "the **unsteered** row. **No success claim survives fidelity > 1.**\n\n" + "\n".join(rows)))

In [ ]:
# [5] TEST 3 — scale sweep. Separates "wrong direction" from "right direction, wrong magnitude".
SCALES = [0.5, 1, 2, 3, 4, 6, 8]
SWEEP = {}
for a in [0.0, 1.0]:
    base = delta_alpha(a)
    for s in SCALES:
        lab = f"α={a:.0f} × {s}"
        SWEEP[lab] = score(s * base, lab, dict(CARDS))   # scored against the same unsteered row
        SWEEP[lab]["alpha"], SWEEP[lab]["scale"] = a, s

rows = ["| α | scale | ‖Δ‖ ÷ one dynamics step | Edit Index ↑ | Target RMSE ↓ | Ghost RMSE ↓ | GT-traj RMSE ↓ | fidelity ↓ |",
        "|---|---|---|---|---|---|---|---|"]
for lab, c in SWEEP.items():
    rows.append(f"| {c['alpha']:.0f} | {c['scale']}× | {c['mag_step']:.2f}× | **{c['edit_index']:+.2f}** | "
                f"{c['target_rmse']:.3f} | {c['ghost_rmse']:.3f} | {c['gt_traj_rmse']:.3f} | {c['fidelity_ratio']:.2f} |")
display(Markdown(f"**Table 3 — Test 3: scale sweep.** A successful edit is "
                 f"**{np.linalg.norm(DH['counterfactual state overwrite'],axis=-1).mean()/STEP_NORM:.1f}× one "
                 f"dynamics step**, and the minimum-norm solutions are far smaller, so magnitude is a live "
                 "alternative explanation for the failure. If the direction is right and only the scale is "
                 "wrong, the Edit Index should rise then fall as scale increases. If the direction is wrong, "
                 "scaling only degrades — watch the fidelity ratio.\n\n" + "\n".join(rows)))

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
CA = {0.0: "#0072B2", 1.0: "#D55E00"}
for a in [0.0, 1.0]:
    ss = [SWEEP[f"α={a:.0f} × {s}"] for s in SCALES]
    lab = "α=0 (Euclidean, existing editor)" if a == 0 else "α=1 (full Mahalanobis)"
    ax[0].plot([c["mag_step"] for c in ss], [c["edit_index"] for c in ss], "-o", color=CA[a], lw=2.0, ms=5, label=lab)
    ax[1].plot([c["mag_step"] for c in ss], [c["fidelity_ratio"] for c in ss], "-o", color=CA[a], lw=2.0, ms=5, label=lab)
    ax[2].plot([c["mag_step"] for c in ss], [c["target_rmse"] for c in ss], "-o", color=CA[a], lw=2.0, ms=5, label=lab)
u = CARDS["unsteered (no edit)"]; orc = CARDS["counterfactual state overwrite (ORACLE)"]
ax[0].axhline(u["edit_index"], color="0.5", ls=":", lw=1.4)
ax[0].annotate(f"unsteered {u['edit_index']:+.2f}", xy=(ax[0].get_xlim()[1], u["edit_index"]), fontsize=7.5,
               color="0.4", ha="right", va="bottom")
ax[0].axhline(orc["edit_index"], color="#009E73", ls="--", lw=1.4)
ax[0].annotate(f"oracle {orc['edit_index']:+.2f}", xy=(ax[0].get_xlim()[1], orc["edit_index"]), fontsize=7.5,
               color="#009E73", ha="right", va="bottom")
ax[0].set_ylabel("Edit Index"); ax[0].set_title("(a) does more magnitude land the edit?", fontsize=10)
ax[1].axhline(1.0, color="#D55E00", ls="--", lw=1.5)
ax[1].annotate("worse than doing nothing ↑", xy=(ax[1].get_xlim()[1], 1.0), fontsize=7.5,
               color="#D55E00", ha="right", va="bottom")
ax[1].set_ylabel("fidelity ratio"); ax[1].set_title("(b) or does it just damage the model?", fontsize=10)
ax[2].axhline(u["target_rmse"], color="0.5", ls=":", lw=1.4)
ax[2].annotate("unsteered", xy=(ax[2].get_xlim()[1], u["target_rmse"]), fontsize=7.5, color="0.4",
               ha="right", va="bottom")
ax[2].set_ylabel("Target RMSE (obs intensity)"); ax[2].set_title("(c) does the object appear at the target?", fontsize=10)
for a_ in ax:
    a_.set_xscale("log"); a_.set_xlabel("‖Δ‖ ÷ one ordinary dynamics step (log scale)")
    a_.legend(fontsize=7.5); a_.grid(alpha=0.3); style_ax(a_)
fig.suptitle("Fig 3 — Test 3: is it the wrong direction, or the right direction at the wrong scale?",
             y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_scale_sweep.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [6] TEST 4 — a LOCAL metric: Σ_hh estimated from nearest neighbours of h0 across trajectories.
# A global Σ assumes one metric everywhere; path-independence and superposition are local properties.
K_NN = 1024                      # > 256 so each local covariance is generically full rank
t0 = time.perf_counter()
Hb_t = torch.tensor(H_BANK, dtype=torch.float32, device=DEVICE)
h0_t = torch.tensor(h0, dtype=torch.float32, device=DEVICE)
nn_idx = torch.cdist(h0_t, Hb_t).topk(K_NN, largest=False).indices.cpu().numpy()   # (N_EVAL, K_NN)

DIR_LOC, EDIT_LOC = {}, {}
loc_cos = {a: [] for a in ALPHAS}
loc_dh = {a: np.zeros_like(h0) for a in ALPHAS}
for i in range(N_EVAL):
    Nb = H_BANK[nn_idx[i]]
    Nb = Nb - Nb.mean(0)
    S_loc = (Nb.T @ Nb) / (K_NN - 1)
    lam_i, V_i = np.linalg.eigh(S_loc)
    lam_i = np.clip(lam_i[::-1], max(lam_i[-1], 1e-300) * 1e-12, None); V_i = V_i[:, ::-1]
    for a in ALPHAS:
        Sa = (V_i * (lam_i ** a)) @ V_i.T
        M = W @ Sa @ W.T
        M = M + EPS_SCALE * np.trace(M) / M.shape[0] * np.eye(M.shape[0])
        d = np.linalg.solve(M, DELTA[i]) @ (W @ Sa)
        loc_dh[a][i] = d
print(f"local metrics ({K_NN}-NN) for {N_EVAL} states in {time.perf_counter()-t0:.0f}s")

rows = ["| metric | α | Δh_true from | cosine ↑ | angle | Edit Index ↑ | ‖Δ‖ ÷ step | fidelity ↓ |",
        "|---|---|---|---|---|---|---|---|"]
LOC_CARDS = {}
for a in ALPHAS:
    lab = f"LOCAL metric Δ_α, α={a:.2f}"
    c = score(loc_dh[a], lab, dict(CARDS)); LOC_CARDS[lab] = c
    for onm, dh in DH.items():
        cc = cos_to(loc_dh[a], dh)
        rows.append(f"| local ({K_NN}-NN) | {a:.2f} | {onm} | **{cc.mean():+.3f}** | "
                    f"{np.degrees(np.arccos(np.clip(cc.mean(),-1,1))):.0f}° | "
                    f"{c['edit_index']:+.2f} | {c['mag_step']:.2f}× | {c['fidelity_ratio']:.2f} |")
for a in ALPHAS:
    gl = CARDS[("readout injection — Euclidean, α=0 (the existing failing editor)" if a == 0
                else f"metric-corrected Δ_α, α={a:.2f}" + (" (full Mahalanobis)" if a == 1 else ""))]
    for onm, dh in DH.items():
        cc = cos_to(delta_alpha(a), dh)
        rows.append(f"| global | {a:.2f} | {onm} | **{cc.mean():+.3f}** | "
                    f"{np.degrees(np.arccos(np.clip(cc.mean(),-1,1))):.0f}° | "
                    f"{gl['edit_index']:+.2f} | {gl['mag_step']:.2f}× | {gl['fidelity_ratio']:.2f} |")
display(Markdown(f"**Table 4 — Test 4: local versus global metric.** Local `Σ_hh` from the **{K_NN} nearest "
                 "neighbours of `h₀` in hidden-state space across all trajectories in the bank** (not a temporal "
                 "window). Everything else is identical to Tests 1–2.\n\n" + "\n".join(rows)))

fig, ax = plt.subplots(1, 2, figsize=(14.5, 4.5))
for onm in DH:
    ax[0].plot(ALPHAS, [cos_to(delta_alpha(a), DH[onm]).mean() for a in ALPHAS], "-o", color=CO[onm], lw=2.0,
               ms=5, label=f"global · {onm}")
    ax[0].plot(ALPHAS, [cos_to(loc_dh[a], DH[onm]).mean() for a in ALPHAS], "--s", color=CO[onm], lw=2.0,
               ms=5, label=f"local ({K_NN}-NN) · {onm}")
ax[0].axhline(0, color="0.4", ls=":", lw=1.0)
ax[0].set_xlabel("α"); ax[0].set_ylabel("cosine to Δh_true")
ax[0].set_title("(a) direction agreement — local vs global metric", fontsize=10)
ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3); style_ax(ax[0])

gl_idx = [CARDS[("readout injection — Euclidean, α=0 (the existing failing editor)" if a == 0
                 else f"metric-corrected Δ_α, α={a:.2f}" + (" (full Mahalanobis)" if a == 1 else ""))]["edit_index"]
          for a in ALPHAS]
ax[1].plot(ALPHAS, gl_idx, "-o", color="#0072B2", lw=2.0, ms=5, label="global metric")
ax[1].plot(ALPHAS, [LOC_CARDS[f"LOCAL metric Δ_α, α={a:.2f}"]["edit_index"] for a in ALPHAS], "--s",
           color="#D55E00", lw=2.0, ms=5, label=f"local metric ({K_NN}-NN)")
u = CARDS["unsteered (no edit)"]
ORC_CARD = CARDS["counterfactual state overwrite (ORACLE)"]
ax[1].axhline(u["edit_index"], color="0.5", ls=":", lw=1.4)
ax[1].annotate(f"unsteered {u['edit_index']:+.2f}", xy=(1.0, u["edit_index"]), fontsize=7.5, color="0.4",
               ha="right", va="bottom")
ax[1].axhline(ORC_CARD["edit_index"], color="#009E73", ls="--", lw=1.4)
ax[1].annotate(f"oracle {ORC_CARD['edit_index']:+.2f}", xy=(1.0, ORC_CARD["edit_index"]), fontsize=7.5, color="#009E73",
               ha="right", va="top")
ax[1].set_xlabel("α"); ax[1].set_ylabel("Edit Index"); ax[1].set_ylim(-1.05, 1.05)
ax[1].set_title("(b) does either metric produce a working edit?", fontsize=10)
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3); style_ax(ax[1])
fig.suptitle("Fig 4 — Test 4: does a locally-estimated metric do better than a global one?", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_local_metric.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [7] Fig 5 — observation waterfalls (canonical spec, one helper), including the degenerate large-scale arm.
N_CTX = 6
DARK, TXT, TICK, EDIT_C = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850"
TARGET_C, GHOST_C = "#00E676", "#FF5252"
CTX = edits.obs[:N_EVAL, ef - N_CTX:ef, :].astype(np.float32)
def _cx(m):
    i = np.where(m)[0]; return i.mean() if i.size else np.nan
tgt_cx = np.array([_cx(ZONES.target[i]) for i in range(N_EVAL)])
gho_cx = np.array([_cx(ZONES.ghost[i]) for i in range(N_EVAL)])

def waterfall_grid(col_titles, col_bodies, samples, suptitle, fname, ctx, dpi=115):
    """Canonical spec: gray on dark; N_CTX NOISY context frames above a dashed edit line; below it EVERY
    column shows its OWN free-run from step 0 (which decodes sim frame `ef`). No shared teacher-forced row."""
    ncol = len(col_titles)
    fig, axes = plt.subplots(len(samples), ncol, figsize=(2.95 * ncol, 3.4 * len(samples)),
                             squeeze=False, facecolor=DARK)
    for r, smp in enumerate(samples):
        for c in range(ncol):
            a = axes[r][c]; a.set_facecolor(DARK)
            panel = np.clip(np.concatenate([ctx[smp], col_bodies[c][smp]], 0), 0, 1)
            a.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            for sp in a.spines.values(): sp.set_edgecolor(TICK)
            a.axhline(N_CTX - 0.5, color=EDIT_C, lw=1.4, ls="--", alpha=0.95)
            if not np.isnan(tgt_cx[smp]): a.axvline(tgt_cx[smp], color=TARGET_C, lw=1.5, alpha=0.9)
            if not np.isnan(gho_cx[smp]): a.axvline(gho_cx[smp], color=GHOST_C, ls="--", lw=1.5, alpha=0.9)
            if r == 0: a.set_title(col_titles[c], fontsize=7, color=TXT)
            if c == 0:
                a.set_ylabel(f"sample {smp}\nsim frame", fontsize=8, color=TXT)
                a.set_yticks([0, N_CTX, N_CTX + 7, N_CTX + 14])
                a.set_yticklabels([ef - N_CTX, ef, ef + 7, ef + 14], fontsize=7)
            else: a.set_yticks([])
            a.set_xlabel("ray", fontsize=8, color=TXT); a.tick_params(colors=TICK, labelsize=7)
    fig.legend(handles=[Line2D([0],[0], color=TARGET_C, lw=2.2, label="target location (where the object should be)"),
                        Line2D([0],[0], color=GHOST_C, ls="--", lw=2.2, label="ghost location (where it was before the edit)"),
                        Line2D([0],[0], color=EDIT_C, ls="--", lw=2.2,
                               label=f"edit applied here — {N_CTX} noisy context frames above; every row below is "
                                     f"that column's OWN free-run, step 0 = sim frame {ef}")],
               loc="upper center", ncol=1, fontsize=8.5, frameon=False, labelcolor=TXT, bbox_to_anchor=(0.5, 0.965))
    fig.suptitle(suptitle, y=1.0, fontsize=10.5, color=TXT)
    fig.tight_layout(rect=[0, 0, 1, 0.885])
    fig.savefig(f"{OUT}/{fname}", dpi=dpi, bbox_inches="tight", facecolor=DARK)
    display(fig); plt.close(fig); print("saved", fname)

teleport = np.linalg.norm(TGT[np.arange(N_EVAL), oe] - PRE[np.arange(N_EVAL), oe], axis=-1)
SAMPLES = list(np.argsort(teleport * (ZONES.ghost.sum(1) >= 3))[::-1][:3])
print("waterfall samples (largest teleports):", SAMPLES)

# The best scale must be chosen among arms that did NOT degrade the model — picking by Edit Index alone
# selects ×8, whose fidelity is 1.57, i.e. the garbage arm, and duplicates the degenerate column.
_ok = [s for s in SCALES if SWEEP[f"α=1 × {s}"]["fidelity_ratio"] <= 1.0]
best_scale = max(_ok, key=lambda s: SWEEP[f"α=1 × {s}"]["edit_index"])
print(f"best NON-DEGRADING scale for α=1: ×{best_scale} "
      f"(index {SWEEP[f'α=1 × {best_scale}']['edit_index']:+.2f}, "
      f"fidelity {SWEEP[f'α=1 × {best_scale}']['fidelity_ratio']:.2f}); "
      f"×8 is shown beside it as the degenerate extreme (fidelity {SWEEP['α=1 × 8']['fidelity_ratio']:.2f})")
PANELS = [("unsteered (no edit)", CARDS["unsteered (no edit)"]),
          ("readout injection — Euclidean α=0\n(the existing failing editor)",
           CARDS["readout injection — Euclidean, α=0 (the existing failing editor)"]),
          ("metric-corrected α=0.5", CARDS["metric-corrected Δ_α, α=0.50"]),
          ("metric-corrected α=1 (full Mahalanobis)", CARDS["metric-corrected Δ_α, α=1.00 (full Mahalanobis)"]),
          (f"α=1 at its best NON-DEGRADING scale (×{best_scale})", SWEEP[f"α=1 × {best_scale}"]),
          ("α=1 at ×8 (the degenerate extreme)", SWEEP["α=1 × 8"]),
          (f"LOCAL metric α=1 ({K_NN}-NN)", LOC_CARDS["LOCAL metric Δ_α, α=1.00"]),
          ("counterfactual state overwrite (ORACLE)", CARDS["counterfactual state overwrite (ORACLE)"])]
titles = ["GT (sim)\nthe true post-edit world"] + [
    f"{n}\nindex {c['edit_index']:+.2f} · ghost {c['ghost_rmse']:.2f} · fidelity {c['fidelity_ratio']:.2f}"
    for n, c in PANELS]
waterfall_grid(titles, [gt_roll] + [c["roll"] for _, c in PANELS], SAMPLES,
               "Fig 5 — what the metric-corrected editors actually generate",
               "fig5_metric_waterfall.png", ctx=CTX)

---
## Summary

In [ ]:
# [8] Computed summary, following the interpretation guide agreed before the run.
print("=" * 100)
print("METRIC-CORRECTED EDITS — is the probe's row space J whitened, and does un-whitening recover it?")
print("=" * 100)
print(f"\nGATE: condition number of Σ_hh = {cond:.3g}  -> "
      + ("PASSED (>20): strongly anisotropic, whitening could matter"
         if cond > 20 else "FAILED (<20): near-isotropic, hypothesis likely wrong"))
print(f"      state distribution occupies {n90}/{n95}/{n99} dims at 90/95/99% variance, of {Hd}.")

base_c = {o: T1[(o, "constraint-satisfying  Δ_α", 0.0)]["cos"] for o in DH}
best_c = {o: max(ALPHAS, key=lambda a: T1[(o, "constraint-satisfying  Δ_α", a)]["cos"]) for o in DH}
print("\nTEST 1 — DIRECTION:")
for o in DH:
    bc = T1[(o, "constraint-satisfying  Δ_α", best_c[o])]["cos"]
    print(f"  {o:<32s} α=0 cos {base_c[o]:+.3f} -> best α={best_c[o]:.2f} cos {bc:+.3f} "
          f"({np.degrees(np.arccos(np.clip(bc,-1,1))):.0f}°)  = {bc - base_c[o]:+.3f}")
    print(f"      subspace mass fraction {T1[(o,'constraint-satisfying  Δ_α',0.0)]['mass']:.3f} -> "
          f"{T1[(o,'constraint-satisfying  Δ_α',best_c[o])]['mass']:.3f}  (chance {CHANCE4:.3f})")
jump = max(T1[(o, "constraint-satisfying  Δ_α", best_c[o])]["cos"] - base_c[o] for o in DH)
print(f"  -> " + ("BIG cosine jump: un-whitening does move the direction toward the true edit."
                  if jump > 0.15 else "NO meaningful cosine jump."))

u = CARDS["unsteered (no edit)"]; orc = CARDS["counterfactual state overwrite (ORACLE)"]
a1 = CARDS["metric-corrected Δ_α, α=1.00 (full Mahalanobis)"]
a0 = CARDS["readout injection — Euclidean, α=0 (the existing failing editor)"]
print(f"\nTEST 2 — CLOSED-FORM EDITOR (apples-to-apples, only the metric differs):")
print(f"  unsteered {u['edit_index']:+.2f} | Euclidean α=0 {a0['edit_index']:+.2f} | "
      f"Mahalanobis α=1 {a1['edit_index']:+.2f} | oracle {orc['edit_index']:+.2f}")
print(f"  fidelity: α=0 {a0['fidelity_ratio']:.2f}, α=1 {a1['fidelity_ratio']:.2f} "
      f"(>1 = worse than doing nothing)")

bs = max(SWEEP.values(), key=lambda c: c["edit_index"])
ok = [c for c in SWEEP.values() if c["fidelity_ratio"] <= 1.0]
bok = max(ok, key=lambda c: c["edit_index"]) if ok else None
print(f"\nTEST 3 — SCALE SWEEP (a successful edit is "
      f"{orc['mag_step']:.1f}× one dynamics step):")
print(f"  best index overall: α={bs['alpha']:.0f} ×{bs['scale']} -> {bs['edit_index']:+.2f} "
      f"at {bs['mag_step']:.1f}× a step, fidelity {bs['fidelity_ratio']:.2f}"
      + ("  <-- DEGRADED, not a success" if bs["fidelity_ratio"] > 1 else ""))
if bok is not None:
    print(f"  best NON-DEGRADING: α={bok['alpha']:.0f} ×{bok['scale']} -> {bok['edit_index']:+.2f}, "
          f"fidelity {bok['fidelity_ratio']:.2f}")

print(f"\nTEST 4 — LOCAL vs GLOBAL METRIC ({K_NN}-NN):")
for a in [1.0]:
    l = LOC_CARDS[f"LOCAL metric Δ_α, α={a:.2f}"]
    for o in DH:
        print(f"  α={a:.0f} vs {o:<32s} global cos {cos_to(delta_alpha(a), DH[o]).mean():+.3f} | "
              f"local cos {cos_to(loc_dh[a], DH[o]).mean():+.3f}")
    print(f"  α={a:.0f} Edit Index: global {a1['edit_index']:+.2f} | local {l['edit_index']:+.2f} "
          f"(fidelity {l['fidelity_ratio']:.2f})")

print("\n" + "-" * 100)
print("VERDICT (per the interpretation guide fixed before the run):")
works = max(c["edit_index"] for c in list(CARDS.values()) + list(SWEEP.values()) + list(LOC_CARDS.values())
            if c["fidelity_ratio"] <= 1.0 and c is not orc)
if jump > 0.15 and works > 0:
    print("  Big cosine jump AND working edits -> MECHANISM CONFIRMED.")
elif jump > 0.15:
    print("  Cosine jump but NO working edit -> points at magnitude or manifold reachability;")
    print("  the scale sweep above adjudicates.")
elif cond > 20:
    print("  NO cosine jump despite a large condition number -> the WHITENING HYPOTHESIS IS FALSIFIED.")
    print("  Remaining explanations narrow to the target being unreachable from readout-derived")
    print("  subspaces at all, regardless of the metric used to pick the direction within them.")
else:
    print("  Gate failed; the hypothesis was not testable on this model.")
print("=" * 100)
print("figures:", sorted(os.listdir(OUT)))

### Current results (updated 2026-08-05)

*GRU `runs/controls/H256`; bank = 78,000 aligned `test` states; N=256 held-out edits. This block is the only
place in the notebook where results live.*

**Verdict against the pre-registered guide: cosine jump, no working edit.** The whitening mechanism is **real
and measurable** — it is the first probe-derived subspace in this thread to be meaningfully *enriched* rather
than at chance — but correcting for it does not produce an edit. The scale sweep shows magnitude is not the
missing ingredient either.

**Gate — passed, decisively.** `Σ_hh` has condition number **1.79 × 10⁴** (λ_max 4.72, λ_min 2.6e-4), and the
states occupy **40 / 76 / 173** dims at 90 / 95 / 99% variance. Strongly anisotropic, so `span(Σ⁻¹J)` and
`span(J)` can differ a great deal. This also **corroborates the derivation independently**: `../iterative_probing/`
found the linear position code sits in below-average-variance directions, which is exactly what `Σ⁻¹` does to a
least-squares probe.

**Test 1 — the direction really does improve, by a lot in relative terms.**

| Δh_true from | α=0 (existing editor) | α=1 (full un-whitening) |
|---|---|---|
| counterfactual overwrite | cos **+0.079** (85°), mass 0.098 = **0.78× chance** | cos **+0.236** (76°), mass 0.380 = **3.04× chance** |
| freeze-time forcing | cos **+0.058** (87°), mass 0.072 = **0.58× chance** | cos **+0.232** (77°), mass 0.366 = **2.93× chance** |

Monotone in α, and the two families (`Δ_α` and the literal `Σ^α W⁺δ`) agree to ±0.003 throughout, so the
constraint-satisfying reformulation costs nothing. The reachable subspace moves from **below chance** to
**3× chance** — but note the absolute scale: 76° is still nearly orthogonal.

**Test 2 — a real, non-degrading improvement that is still not an edit.** Only the metric differs from the
existing failing editor:

| editor | ‖Δ‖ ÷ dynamics step | Edit Index | Target RMSE | Ghost RMSE | fidelity |
|---|---|---|---|---|---|
| unsteered | — | −0.67 | 0.488 | 0.589 | 1.00 |
| Euclidean α=0 *(existing)* | 0.34× | −0.65 | 0.479 | 0.586 | 1.00 |
| **Mahalanobis α=1** | 1.13× | **−0.51** | 0.432 | 0.534 | **0.98** |
| counterfactual overwrite *(oracle)* | 3.75× | +0.68 | 0.098 | 0.099 | 0.56 |

**+0.14 index points at zero fidelity cost, training-free.** For scale: the *heavy fine-tuning* arm of the
trained-editability thread bought +0.13 while costing 13% of next-step prediction. This is the **best
training-free structural editor the thread has produced** — and it is still deep on the unedited side.

**Test 3 — the scale sweep adjudicates, and magnitude is not the answer.** For α=1 there is a genuine optimum,
and it sits almost exactly where the displacement matches the oracle's:

| scale | ‖Δ‖ ÷ step | Edit Index | Target RMSE | Ghost RMSE | fidelity |
|---|---|---|---|---|---|
| ×1 | 1.13× | −0.51 | 0.432 | 0.534 | 0.98 |
| **×2** | 2.26× | **−0.33** | 0.394 | 0.497 | **0.99** |
| ×3 | 3.39× | −0.21 | **0.380** | **0.483** | 1.05 |
| ×4 | 4.53× | −0.13 | 0.392 | 0.494 | 1.14 |
| ×8 | 9.05× | +0.01 | 0.627 | 0.726 | **1.57** |

Target and Ghost RMSE both **minimise at ×3**, where ‖Δ‖ = 3.39× a dynamics step against the oracle's 3.75× —
the right-direction/right-magnitude combination is identifiable, and it is not where the min-norm solution puts
you. But fidelity crosses 1 at ×3, so the best *legitimate* arm is **×2 → −0.33**, which is **25% of the
oracle's gain**. Past that the index keeps rising only by degrading: ×8 reaches +0.01 with fidelity 1.57 and
every zone worse than unsteered — Fig 5's ×8 column is bright striped garbage. **So the answer is neither
"wrong direction" nor "wrong magnitude" alone.** Fixing both gets a quarter of the way.

Scaling the *Euclidean* direction is much weaker: α=0 at ×8 reaches only −0.45 with Target RMSE barely moving
(0.479 → 0.447), so the metric correction contributes beyond magnitude. At matched displacement (~2.2× a step),
α=1 gives −0.33 / Target 0.394 versus α=0's ≈−0.50 / Target ≈0.45.

**Test 4 — the local metric is *worse*, once magnitude is controlled.** Local `Σ_hh` from 1024 nearest
neighbours gives cos **+0.143** at α=1 against the global metric's **+0.236**, and peaks at α=0.5 (0.177) before
declining. Its Edit Index of −0.38 *looks* better than the global −0.51, but that is entirely a magnitude
effect: the local editor produces ‖Δ‖ = 2.27× a step versus the global's 1.13×, and at matched displacement the
global metric wins (global ×2 → −0.33 at 2.26× versus local → −0.38 at 2.27×). **A locally-estimated metric buys
nothing here**, which is itself informative: the anisotropy that matters is a global property of the state
distribution, not local curvature.

**Where this leaves the hypothesis.** Partially confirmed and insufficient. The whitening account is
*mechanistically real* — it explains why probe-derived directions have looked orthogonal to successful edits,
and correcting it produces the thread's largest training-free gain. But recovering `J` is not enough to edit.
Reading this beside the two neighbouring results:

| method | fraction of `Δh_true` captured | fraction of the oracle's Edit Index gain |
|---|---|---|
| tangent-constrained projection (22-dim local PCA) | 57% | ~33% |
| projection onto the 116-dim position code | 57% | 33% |
| metric-corrected direction, best legitimate scale | ~24–38% | 25% |

Three unrelated constructions, and the recovered effect is consistently **well below** the recovered fraction of
the vector. That is the all-or-nothing signature again, now with a graded version: partial capture buys
sub-proportional effect.

**Caveats.** One model, one seed, position probe only. `Σ_hh` is estimated over all timesteps including the
early frames where the filter has not converged. The α grid is coarse (5 points) and the scale grid coarser
(7 points); the ×2–×3 optimum is bracketed but not resolved. The local metric uses a single `k = 1024`, with no
sweep over neighbourhood size — a smaller `k` would give a more local (and lower-rank) estimate and was not
tried.